# **WEEK 4 — Spectral Analysis and Frequency Domain Methods**

>Jerald B. Bongalos | PhD in Data Science | Asian Institute of Management

---

*Reference:*
>Shumway, R. H., & Stoffer, D. S. *Time Series Analysis and Its Applications*
>(Ch 4.1–4.7: Spectral Analysis and Filtering)

---

**Purpose of this Notebook**

This notebook develops the **frequency domain perspective** on time series analysis.
The time domain (ACF, ARMA) asks: *how does the present depend on the past?*
The frequency domain asks: *what periodic components make up the signal?*

Both perspectives describe the same underlying process — they are connected by
the **Wiener–Khinchin theorem** (spectral density ↔ autocovariance).

**The goal is to understand:**

- what the spectral density function represents and why it exists,
- how the periodogram estimates spectral power (and why it is inconsistent),
- how spectral smoothing produces consistent estimators,
- how linear filtering modifies the spectrum,
- and how the frequency domain connects to everything learned in Weeks 1–3.

**Learning Outcomes**

By the end of this notebook, you should be able to:

1. State the spectral representation theorem and the Wiener–Khinchin relation
2. Compute and interpret the periodogram
3. Explain why the raw periodogram is an inconsistent estimator
4. Apply spectral smoothing (Daniell kernel, tapering)
5. Analyze the spectral signature of AR, MA, and ARMA processes
6. Explain how linear filters modify the spectrum (transfer function)
7. Distinguish spectral peaks from noise

---

**Notebook Structure**

| Part | Topic | Type |
|------|-------|------|
| 1 | From Time Domain to Frequency Domain | Theory |
| 2 | Spectral Density Function | Theory + Simulation |
| 3 | The Periodogram | Theory + Simulation |
| 4 | Spectral Smoothing and Consistent Estimation | Theory + Simulation |
| 5 | Spectral Signatures of ARMA Processes | Theory + Simulation |
| 6 | Linear Filters and the Transfer Function | Theory + Simulation |
| 7 | Synthesis (Exam-Ready) | Summary |
| 8 | Self-Test Questions | Exam Preparation |


In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import periodogram as scipy_periodogram, welch
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima_process import arma_generate_sample

np.random.seed(123)

# --- Shared AR(1) / MA(1) simulators ---
def simulate_ar1(phi, n=500, sigma=1.0, burnin=200):
    eps = np.random.normal(0, sigma, n + burnin)
    x = np.zeros(n + burnin)
    for t in range(1, n + burnin):
        x[t] = phi * x[t-1] + eps[t]
    return x[burnin:]

def simulate_ma1(theta, n=400, sigma=1.0):
    eps = np.random.normal(0, sigma, n + 1)
    return eps[1:] + theta * eps[:-1]

def simulate_arma11(phi, theta, n=500, sigma=1.0, burnin=200):
    eps = np.random.normal(0, sigma, n + burnin + 1)
    x = np.zeros(n + burnin + 1)
    for t in range(1, n + burnin + 1):
        x[t] = phi * x[t-1] + eps[t] + theta * eps[t-1]
    return x[burnin+1:]

# --- Raw periodogram helper ---
def compute_periodogram(x):
    """Compute raw periodogram: I(f_j) = |DFT|^2 / n."""
    x = np.asarray(x, dtype=float)
    x = x - np.mean(x)
    n = len(x)
    freqs = np.fft.rfftfreq(n, d=1.0)
    Fk = np.fft.rfft(x)
    Pxx = (np.abs(Fk)**2) / n
    return freqs, Pxx

print("Setup complete.")


## **PART 1 — From Time Domain to Frequency Domain**

> **Why this part matters**
>
> The time domain describes dependence through autocovariances: *how does $X_t$ relate to $X_{t-k}$?*
> The frequency domain describes the **same structure** through periodic components:
> *how much power is concentrated at each frequency?*
>
> These are not competing views — they are **dual representations** of the same process.

---

### **1.1 The Core Idea**

Any weakly stationary process can be decomposed into a sum of sinusoidal components
at different frequencies, each with a specific amplitude and phase.

- **Low frequency** ($\omega \approx 0$): slow, trend-like variation
- **High frequency** ($\omega \approx 0.5$ cycles/unit): rapid oscillation
- **Intermediate frequencies**: periodic patterns (e.g., seasonality)

---

### **1.2 Frequency and Period**

If the sampling interval is 1 (e.g., 1 month), then:

$$
\text{frequency } \omega \in [0, 0.5] \text{ cycles per unit time}
$$

$$
\text{period} = \frac{1}{\omega} \text{ units}
$$

For monthly data:
- $\omega = 1/12 \approx 0.083$: annual cycle (period = 12 months)
- $\omega = 1/6 \approx 0.167$: semi-annual cycle (period = 6 months)
- $\omega = 0.5$: Nyquist frequency (fastest detectable oscillation)

---

### **1.3 Why Bother with the Frequency Domain?**

The frequency domain is essential when:

- Detecting **hidden periodicities** (seasonal cycles, astronomical periods)
- Designing **filters** (removing trend, isolating cycles)
- Understanding **aliasing** and sampling effects
- Connecting to **physical mechanisms** that operate at specific frequencies
- Diagnosing model adequacy in ways ACF cannot

> **Exam-safe statement:** "The time and frequency domains are equivalent descriptions
> of a stationary process, connected by the Wiener–Khinchin theorem."


## **PART 2 — Spectral Density Function**

> **Why this part matters**
>
> The spectral density function $f(\omega)$ tells us how **variance is distributed across frequencies**.
> It is the frequency-domain analog of the autocovariance function $\gamma(k)$.

---

### **2.1 The Wiener–Khinchin Theorem**

For a weakly stationary process with absolutely summable autocovariances, the **spectral density** is:

$$
f(\omega) = \sum_{h=-\infty}^{\infty} \gamma(h)\, e^{-2\pi i \omega h} \;=\; \gamma(0) + 2\sum_{h=1}^{\infty} \gamma(h) \cos(2\pi \omega h)
$$

The **inverse** relationship recovers autocovariances from the spectrum:

$$
\gamma(h) = \int_{-1/2}^{1/2} f(\omega)\, e^{2\pi i \omega h}\, d\omega
$$

At $h = 0$:

$$
\gamma(0) = \mathrm{Var}(X_t) = \int_{-1/2}^{1/2} f(\omega)\, d\omega
$$

> The total area under the spectral density equals the variance. The spectrum decomposes
> variance by frequency.

---

### **2.2 Key Properties**

- $f(\omega) \geq 0$ for all $\omega$ (variance cannot be negative)
- $f(\omega) = f(-\omega)$ (symmetric for real-valued processes)
- $\int f(\omega)\, d\omega = \gamma(0)$ (total variance)
- Sharp peaks in $f(\omega)$ indicate dominant periodic components
- Flat spectrum = white noise ($f(\omega) = \sigma^2$ for all $\omega$)

---

### **2.3 Spectral Density of White Noise**

If $\varepsilon_t \sim \mathrm{WN}(0, \sigma^2)$, then $\gamma(0) = \sigma^2$ and $\gamma(h) = 0$ for $h \neq 0$:

$$
f_{\varepsilon}(\omega) = \sigma^2 \quad \text{for all } \omega
$$

White noise has a **flat spectrum** — equal power at all frequencies. This is why it is called "white" (by analogy with white light containing all frequencies).


In [ ]:
# ============================================================
# PART 2: Spectral Density — White Noise vs Signal + Noise
# ============================================================

np.random.seed(123)
n = 512

# --- A) White noise: flat spectrum ---
wn = np.random.normal(0, 1, n)

# --- B) Signal: two sinusoids buried in noise ---
t = np.arange(n)
signal = 2.0 * np.sin(2*np.pi*t/12) + 1.5 * np.cos(2*np.pi*t/40)
x_signal = signal + np.random.normal(0, 1.5, n)

# --- C) Seasonal monthly series (PM2.5 style) ---
season = 1.5 * np.sin(2*np.pi*t/12) + 0.8 * np.cos(2*np.pi*t/6)
x_season = season + 0.05 * t / n + np.random.normal(0, 0.8, n)

# --- Compute periodograms ---
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for row, (label, series) in enumerate([
    ("A) White Noise", wn),
    ("B) Two Sinusoids + Noise", x_signal),
    ("C) Seasonal Series", x_season)
]):
    # Time plot
    axes[row, 0].plot(series[:200], linewidth=0.8, color="#2c3e50")
    axes[row, 0].set_title(f"{label} — Time Series", fontsize=11)
    axes[row, 0].set_xlabel("$t$")
    axes[row, 0].grid(True, alpha=0.3)

    # Periodogram
    freqs, Pxx = compute_periodogram(series)
    axes[row, 1].plot(freqs[1:], Pxx[1:], linewidth=0.8, color="#2c3e50")
    axes[row, 1].set_title(f"{label} — Periodogram", fontsize=11)
    axes[row, 1].set_xlabel("Frequency (cycles/unit)")
    axes[row, 1].set_ylabel("Power")
    axes[row, 1].grid(True, alpha=0.3)

    # Mark known frequencies
    if row == 1:
        axes[row, 1].axvline(1/12, color="red", linestyle="--", alpha=0.7, label="$\\omega=1/12$")
        axes[row, 1].axvline(1/40, color="blue", linestyle="--", alpha=0.7, label="$\\omega=1/40$")
        axes[row, 1].legend(fontsize=8)
    elif row == 2:
        axes[row, 1].axvline(1/12, color="red", linestyle="--", alpha=0.7, label="Annual ($\\omega=1/12$)")
        axes[row, 1].axvline(1/6, color="orange", linestyle="--", alpha=0.7, label="Semi-annual ($\\omega=1/6$)")
        axes[row, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Observations:")
print("  A) White noise: periodogram is noisy but roughly flat (no dominant frequency)")
print("  B) Two peaks visible at omega=1/12 and omega=1/40 (buried sinusoids detected)")
print("  C) Strong annual peak at omega=1/12, weaker semi-annual at omega=1/6")
print("\nThe periodogram reveals hidden periodicities that ACF may not show clearly.")


## **PART 3 — The Periodogram**

> **Why this part matters**
>
> The periodogram is the **natural estimator** of the spectral density.
> It is computed from the DFT of the observed series. However, it has a critical flaw:
> it is an **inconsistent estimator** — its variance does not shrink as $n \to \infty$.

---

### **3.1 Definition**

The periodogram at Fourier frequency $\omega_j = j/n$ is:

$$
I(\omega_j) = \frac{1}{n}\left|\sum_{t=1}^{n} X_t\, e^{-2\pi i \omega_j t}\right|^2 = \frac{|d(\omega_j)|^2}{n}
$$

where $d(\omega_j)$ is the Discrete Fourier Transform (DFT) at frequency $\omega_j$.

Equivalently, the periodogram can be expressed via autocovariances:

$$
I(\omega_j) = \sum_{h=-(n-1)}^{n-1} \hat{\gamma}(h)\, e^{-2\pi i \omega_j h}
$$

---

### **3.2 The Inconsistency Problem**

The periodogram is an **asymptotically unbiased** estimator of $f(\omega)$:

$$
\mathbb{E}[I(\omega_j)] \to f(\omega_j) \quad \text{as } n \to \infty
$$

But its **variance does not decrease** with $n$:

$$
\mathrm{Var}[I(\omega_j)] \approx f(\omega_j)^2 \quad \text{(does not } \to 0 \text{)}
$$

This means:
- Adding more data does **not** make the periodogram smoother
- Each periodogram ordinate is an **exponentially distributed** random variable (for Gaussian processes)
- The raw periodogram is too noisy for reliable inference

> **Exam-critical:** "The periodogram is an inconsistent estimator of the spectral density
> because its variance does not decrease with sample size."

---

### **3.3 What This Means in Practice**

- The periodogram reveals the **right frequencies** (peaks are in the right places)
- But the **amplitude** at each frequency is unreliable
- Smoothing is required for consistent spectral estimation (Part 4)
- Despite inconsistency, the periodogram is still the starting point for all spectral analysis


In [ ]:
# ============================================================
# PART 3: Periodogram Inconsistency Demonstration
# ============================================================
# Key demo: increasing n does NOT make the periodogram smoother.
# We overlay periodograms from multiple realizations to show variance.
# ============================================================

np.random.seed(42)

# --- AR(1) phi=0.7: true spectral density has a known form ---
phi = 0.7
sigma = 1.0

def ar1_spectral_density(phi, sigma, freqs):
    """True spectral density of AR(1): f(w) = sigma^2 / |1 - phi*e^{-2pi*i*w}|^2."""
    omega = 2 * np.pi * freqs
    denom = np.abs(1 - phi * np.exp(-1j * omega))**2
    return sigma**2 / denom

# --- Show periodograms at different n ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for col, n_val in enumerate([64, 256, 1024]):
    x = simulate_ar1(phi, n=n_val, sigma=sigma)
    freqs, Pxx = compute_periodogram(x)
    f_true = ar1_spectral_density(phi, sigma, freqs)

    axes[col].plot(freqs[1:], Pxx[1:], linewidth=0.5, alpha=0.7, color="#95a5a6", label="Periodogram")
    axes[col].plot(freqs[1:], f_true[1:], linewidth=2.5, color="#e74c3c", label="True $f(\\omega)$")
    axes[col].set_title(f"$n = {n_val}$", fontsize=12)
    axes[col].set_xlabel("Frequency")
    axes[col].set_ylabel("Power")
    axes[col].legend(fontsize=8)
    axes[col].grid(True, alpha=0.3)

plt.suptitle("Periodogram Inconsistency: Increasing $n$ Does NOT Reduce Variance",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# --- Multiple realizations overlay (n=256) ---
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
n_val = 256
for rep in range(20):
    x = simulate_ar1(phi, n=n_val, sigma=sigma)
    freqs, Pxx = compute_periodogram(x)
    ax.plot(freqs[1:], Pxx[1:], linewidth=0.4, alpha=0.3, color="#3498db")

f_true = ar1_spectral_density(phi, sigma, freqs)
ax.plot(freqs[1:], f_true[1:], linewidth=3, color="#e74c3c", label="True $f(\\omega)$")
ax.set_title(f"20 Periodograms from AR(1) $\\phi={phi}$ ($n={n_val}$)", fontsize=12)
ax.set_xlabel("Frequency")
ax.set_ylabel("Power")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observation:")
print("  The 20 periodograms scatter wildly around the true spectrum.")
print("  More data (larger n) gives finer frequency resolution but NOT less scatter.")
print("  This is the inconsistency problem. Smoothing is needed.")


## **PART 4 — Spectral Smoothing and Consistent Estimation**

> **Why this part matters**
>
> The raw periodogram is inconsistent. To obtain a **consistent** spectral estimator,
> we must smooth — either in the frequency domain (averaging nearby periodogram ordinates)
> or in the time domain (tapering the autocovariances).

---

### **4.1 Frequency-Domain Smoothing (Daniell Kernel)**

The smoothed spectral estimate averages periodogram ordinates in a neighborhood:

$$
\hat{f}(\omega) = \sum_{k=-m}^{m} h_k \cdot I(\omega + k/n)
$$

where $\{h_k\}$ is a symmetric kernel (e.g., uniform weights $h_k = 1/(2m+1)$).

**Bandwidth** $m$ controls the bias-variance tradeoff:
- Larger $m$: smoother estimate (lower variance), but peaks are blurred (higher bias)
- Smaller $m$: preserves peaks (lower bias), but noisier (higher variance)

---

### **4.2 Time-Domain Approach (Lag Window / Tapering)**

Alternatively, apply a **taper** (window function) to the sample autocovariances before computing the DFT:

$$
\hat{f}(\omega) = \sum_{h=-(n-1)}^{n-1} w(h)\, \hat{\gamma}(h)\, e^{-2\pi i \omega h}
$$

where $w(h)$ is a weight function that downweights large lags (e.g., Bartlett, Parzen, Daniell).

This is equivalent to frequency-domain smoothing — the window shape determines the effective kernel.

---

### **4.3 Bias-Variance Tradeoff**

| | Small bandwidth | Large bandwidth |
|---|---|---|
| Variance | High (noisy) | Low (smooth) |
| Bias | Low (peaks preserved) | High (peaks blurred) |
| Resolution | Fine | Coarse |

> **Exam-safe statement:** "Consistent spectral estimation requires smoothing.
> The bandwidth controls a bias-variance tradeoff: more smoothing reduces variance
> but may blur spectral peaks."


In [ ]:
# ============================================================
# PART 4: Spectral Smoothing — Daniell Kernel
# ============================================================

np.random.seed(123)

phi = 0.7
sigma = 1.0
n = 512
x = simulate_ar1(phi, n=n, sigma=sigma)

freqs, Pxx = compute_periodogram(x)
f_true = ar1_spectral_density(phi, sigma, freqs)

# --- Daniell smoothing (uniform moving average of periodogram) ---
def daniell_smooth(Pxx, m):
    """Smooth periodogram with Daniell kernel (uniform, width 2m+1)."""
    kernel = np.ones(2*m + 1) / (2*m + 1)
    return np.convolve(Pxx, kernel, mode="same")

# --- Compare different bandwidths ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(freqs[1:], Pxx[1:], linewidth=0.5, color="#95a5a6")
axes[0, 0].plot(freqs[1:], f_true[1:], linewidth=2.5, color="#e74c3c", label="True $f(\\omega)$")
axes[0, 0].set_title("Raw Periodogram (no smoothing)", fontsize=11)
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

for idx, (m_val, ax) in enumerate([(3, axes[0, 1]), (7, axes[1, 0]), (15, axes[1, 1])]):
    Pxx_smooth = daniell_smooth(Pxx, m_val)
    ax.plot(freqs[1:], Pxx_smooth[1:], linewidth=1.5, color="#2c3e50", label=f"Smoothed ($m={m_val}$)")
    ax.plot(freqs[1:], f_true[1:], linewidth=2.5, color="#e74c3c", label="True $f(\\omega)$")
    ax.set_title(f"Daniell Smoothed ($m={m_val}$, bandwidth $= {2*m_val+1}$)", fontsize=11)
    ax.set_xlabel("Frequency")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Spectral Smoothing: Bias-Variance Tradeoff (AR(1), $\\phi={phi}$, $n={n}$)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# --- Also show Welch's method (scipy) ---
fig, ax = plt.subplots(1, 1, figsize=(12, 4))

f_welch, Pxx_welch = welch(x - np.mean(x), fs=1.0, nperseg=128, noverlap=64)
ax.plot(freqs[1:], Pxx[1:], linewidth=0.4, alpha=0.5, color="#95a5a6", label="Raw periodogram")
ax.plot(f_welch, Pxx_welch, linewidth=2, color="#2980b9", label="Welch (nperseg=128)")
ax.plot(freqs[1:], f_true[1:], linewidth=2.5, color="#e74c3c", label="True $f(\\omega)$")
ax.set_title("Welch's Method (Segment-Averaged Periodogram)", fontsize=12)
ax.set_xlabel("Frequency")
ax.set_ylabel("Power spectral density")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observations:")
print("  m=3: still noisy, but peaks emerge")
print("  m=7: good balance — tracks true spectrum closely")
print("  m=15: very smooth but AR peak is blurred")
print("  Welch's method: automatic segment averaging, a practical default")


## **PART 5 — Spectral Signatures of ARMA Processes**

> **Why this part matters**
>
> Every ARMA process has a **closed-form spectral density**. Learning to recognize
> these spectral shapes is the frequency-domain equivalent of reading ACF/PACF patterns.

---

### **5.1 General ARMA Spectral Density**

For ARMA($p,q$) with AR polynomial $\phi(z)$ and MA polynomial $\theta(z)$:

$$
f(\omega) = \sigma^2 \cdot \frac{|\theta(e^{-2\pi i\omega})|^2}{|\phi(e^{-2\pi i\omega})|^2}
$$

---

### **5.2 Specific Cases**

**AR(1):** $\phi(z) = 1 - \phi z$

$$
f(\omega) = \frac{\sigma^2}{|1 - \phi e^{-2\pi i\omega}|^2} = \frac{\sigma^2}{1 - 2\phi\cos(2\pi\omega) + \phi^2}
$$

- $\phi > 0$: peak at $\omega = 0$ (low-frequency dominance, persistence)
- $\phi < 0$: peak at $\omega = 0.5$ (high-frequency dominance, oscillation)

**MA(1):** $\theta(z) = 1 + \theta z$

$$
f(\omega) = \sigma^2 |1 + \theta e^{-2\pi i\omega}|^2 = \sigma^2(1 + 2\theta\cos(2\pi\omega) + \theta^2)
$$

- $\theta > 0$: peak at $\omega = 0$ (low-frequency amplification)
- $\theta < 0$: peak at $\omega = 0.5$ (high-frequency amplification)

**ARMA(1,1):** Combines both effects.

---

### **5.3 The Spectral Identification Table**

| Process | Spectral shape |
|---|---|
| White noise | Flat: $f(\omega) = \sigma^2$ |
| AR(1), $\phi > 0$ | Peak at $\omega = 0$, monotonic decrease |
| AR(1), $\phi < 0$ | Peak at $\omega = 0.5$, monotonic increase |
| MA(1), $\theta > 0$ | Peak at $\omega = 0$ |
| MA(1), $\theta < 0$ | Peak at $\omega = 0.5$ |
| AR(2) with complex roots | Peak at intermediate frequency (pseudo-cycle) |
| Seasonal (period $s$) | Peaks at $\omega = k/s$ for $k = 1, 2, \ldots$ |

> **Exam-safe statement:** "The spectral density of an ARMA process is determined by the
> ratio of MA to AR transfer functions evaluated on the unit circle."


In [ ]:
# ============================================================
# PART 5: Spectral Signatures of AR(1), MA(1), ARMA(1,1), AR(2)
# ============================================================

np.random.seed(123)
n = 1024
freqs_theory = np.linspace(0.001, 0.5, 500)

def ma1_spectral_density(theta, sigma, freqs):
    omega = 2 * np.pi * freqs
    return sigma**2 * np.abs(1 + theta * np.exp(-1j * omega))**2

def arma11_spectral_density(phi, theta, sigma, freqs):
    omega = 2 * np.pi * freqs
    num = np.abs(1 + theta * np.exp(-1j * omega))**2
    den = np.abs(1 - phi * np.exp(-1j * omega))**2
    return sigma**2 * num / den

def ar2_spectral_density(phi1, phi2, sigma, freqs):
    omega = 2 * np.pi * freqs
    den = np.abs(1 - phi1 * np.exp(-1j * omega) - phi2 * np.exp(-2j * omega))**2
    return sigma**2 / den

# --- AR(1) positive vs negative ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1: positive phi / theta
configs_pos = [
    ("AR(1) $\\phi=0.8$", ar1_spectral_density(0.8, 1.0, freqs_theory),
     simulate_ar1(0.8, n=n)),
    ("MA(1) $\\theta=0.8$", ma1_spectral_density(0.8, 1.0, freqs_theory),
     simulate_ma1(0.8, n=n)),
    ("ARMA(1,1) $\\phi=0.6, \\theta=0.4$", arma11_spectral_density(0.6, 0.4, 1.0, freqs_theory),
     simulate_arma11(0.6, 0.4, n=n)),
]

for col, (label, f_theory, x_sim) in enumerate(configs_pos):
    freqs_s, Pxx_s = compute_periodogram(x_sim)
    Pxx_smooth = daniell_smooth(Pxx_s, 5)
    axes[0, col].plot(freqs_s[1:], Pxx_smooth[1:], linewidth=1, color="#95a5a6", alpha=0.8, label="Smoothed pgram")
    axes[0, col].plot(freqs_theory, f_theory, linewidth=2.5, color="#e74c3c", label="True $f(\\omega)$")
    axes[0, col].set_title(label, fontsize=11)
    axes[0, col].legend(fontsize=7)
    axes[0, col].grid(True, alpha=0.3)
    axes[0, col].set_xlabel("Frequency")

# Row 2: negative phi, AR(2) with complex roots, seasonal
configs_neg = [
    ("AR(1) $\\phi=-0.7$", ar1_spectral_density(-0.7, 1.0, freqs_theory),
     simulate_ar1(-0.7, n=n)),
]

# AR(2) with complex roots: phi1=1.0, phi2=-0.75 -> pseudo-cycle
phi1_ar2, phi2_ar2 = 1.0, -0.75
eps_ar2 = np.random.normal(0, 1, n + 200)
x_ar2 = np.zeros(n + 200)
for t in range(2, n + 200):
    x_ar2[t] = phi1_ar2 * x_ar2[t-1] + phi2_ar2 * x_ar2[t-2] + eps_ar2[t]
x_ar2 = x_ar2[200:]

configs_neg.append(("AR(2) complex roots", ar2_spectral_density(phi1_ar2, phi2_ar2, 1.0, freqs_theory), x_ar2))

# Seasonal
t_s = np.arange(n)
x_season = 2 * np.sin(2*np.pi*t_s/12) + np.random.normal(0, 1, n)
f_season_dummy = np.ones(len(freqs_theory))  # placeholder
configs_neg.append(("Seasonal (period=12) + noise", f_season_dummy, x_season))

for col, (label, f_theory, x_sim) in enumerate(configs_neg):
    freqs_s, Pxx_s = compute_periodogram(x_sim)
    Pxx_smooth = daniell_smooth(Pxx_s, 5)
    axes[1, col].plot(freqs_s[1:], Pxx_smooth[1:], linewidth=1, color="#95a5a6", alpha=0.8, label="Smoothed pgram")
    if "Seasonal" not in label:
        axes[1, col].plot(freqs_theory, f_theory, linewidth=2.5, color="#e74c3c", label="True $f(\\omega)$")
    else:
        axes[1, col].axvline(1/12, color="red", linestyle="--", alpha=0.7, label="$\\omega=1/12$")
    axes[1, col].set_title(label, fontsize=11)
    axes[1, col].legend(fontsize=7)
    axes[1, col].grid(True, alpha=0.3)
    axes[1, col].set_xlabel("Frequency")

plt.suptitle("Spectral Signatures of Common Processes", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  AR(1) phi>0: power concentrated at low frequencies (persistence)")
print("  AR(1) phi<0: power concentrated at high frequencies (oscillation)")
print("  MA(1) theta>0: similar low-frequency boost (but bounded spectrum)")
print("  ARMA(1,1): combines AR peak shape with MA modulation")
print("  AR(2) complex roots: sharp peak at intermediate frequency (pseudo-cycle)")
print("  Seasonal: discrete peaks at omega = 1/12, 2/12, ... (harmonics)")


## **PART 6 — Linear Filters and the Transfer Function**

> **Why this part matters**
>
> In time series, **differencing**, **moving averages**, and **seasonal adjustment** are all
> linear filters. The frequency domain reveals exactly what each filter does:
> it multiplies the spectrum by the **squared transfer function**.

---

### **6.1 Linear Filter Definition**

A linear filter transforms $\{X_t\}$ into $\{Y_t\}$:

$$
Y_t = \sum_{j=-\infty}^{\infty} a_j\, X_{t-j} = a(B)\, X_t
$$

where $a(B) = \sum a_j B^j$ is the filter polynomial and $B$ is the backshift operator.

---

### **6.2 Effect on the Spectrum**

If $X_t$ has spectral density $f_X(\omega)$, then the filtered output $Y_t$ has:

$$
\boxed{f_Y(\omega) = |A(\omega)|^2 \cdot f_X(\omega)}
$$

where $A(\omega) = \sum_{j} a_j\, e^{-2\pi i \omega j}$ is the **frequency response** (transfer function).

$|A(\omega)|^2$ is called the **power transfer function** — it tells you how much
the filter amplifies or suppresses each frequency.

---

### **6.3 Important Filters**

**First difference** $\nabla X_t = (1 - B)X_t$:

$$
|A(\omega)|^2 = |1 - e^{-2\pi i\omega}|^2 = 2(1 - \cos 2\pi\omega)
$$

This **suppresses low frequencies** (trend) and amplifies high frequencies.

**Seasonal difference** $\nabla_{12} X_t = (1 - B^{12})X_t$:

$$
|A(\omega)|^2 = |1 - e^{-2\pi i \cdot 12\omega}|^2 = 2(1 - \cos 24\pi\omega)
$$

This has **zeros at** $\omega = k/12$ — it removes seasonal harmonics.

**Moving average** (symmetric, span $2m+1$):

$$
|A(\omega)|^2 = \left|\frac{1}{2m+1}\sum_{j=-m}^{m} e^{-2\pi i\omega j}\right|^2
$$

This is a **low-pass filter** — it smooths by suppressing high frequencies.

> **Exam-safe statement:** "Any linear filter multiplies the spectrum by its power transfer function. Differencing is a high-pass filter; moving average is a low-pass filter."


In [ ]:
# ============================================================
# PART 6: Linear Filters — Transfer Functions and Spectral Effects
# ============================================================

np.random.seed(123)
n = 512
freqs_f = np.linspace(0.001, 0.5, 500)

# --- Power transfer functions ---
omega = 2 * np.pi * freqs_f

# First difference: |1 - e^{-iw}|^2
H_diff1 = 2 * (1 - np.cos(omega))

# Seasonal difference (s=12): |1 - e^{-i*12*w}|^2
H_diff12 = 2 * (1 - np.cos(12 * omega))

# Moving average (span=7)
m_ma = 3  # half-width
H_ma = np.abs(np.sin((2*m_ma+1)*np.pi*freqs_f) / ((2*m_ma+1)*np.sin(np.pi*freqs_f + 1e-10)))**2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(freqs_f, H_diff1, linewidth=2, color="#e74c3c")
axes[0].set_title("First Difference $|1 - e^{-2\\pi i\\omega}|^2$", fontsize=11)
axes[0].set_xlabel("Frequency")
axes[0].set_ylabel("$|A(\\omega)|^2$")
axes[0].annotate("Suppresses\ntrend", xy=(0.02, 0.1), fontsize=9, color="blue")
axes[0].grid(True, alpha=0.3)

axes[1].plot(freqs_f, H_diff12, linewidth=2, color="#8e44ad")
axes[1].set_title("Seasonal Difference $|1 - e^{-24\\pi i\\omega}|^2$", fontsize=11)
axes[1].set_xlabel("Frequency")
for k in range(1, 7):
    axes[1].axvline(k/12, color="red", linestyle=":", alpha=0.4)
axes[1].annotate("Zeros at\n$k/12$", xy=(0.08, 0.3), fontsize=9, color="red")
axes[1].grid(True, alpha=0.3)

axes[2].plot(freqs_f, H_ma, linewidth=2, color="#2980b9")
axes[2].set_title("Moving Average (span=7)", fontsize=11)
axes[2].set_xlabel("Frequency")
axes[2].annotate("Low-pass\n(smooths)", xy=(0.02, 0.8), fontsize=9, color="blue")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Power Transfer Functions of Common Filters", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# --- Apply filters to a signal and show spectral effect ---
t = np.arange(n)
x_orig = simulate_ar1(0.8, n=n) + 1.5 * np.sin(2*np.pi*t/12)

x_diff1 = np.diff(x_orig)
x_diff12 = x_orig[12:] - x_orig[:-12]
x_ma7 = np.convolve(x_orig, np.ones(7)/7, mode="valid")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (label, series) in zip(axes.flat, [
    ("Original: AR(1) + seasonal", x_orig),
    ("First Differenced: $\\nabla X_t$", x_diff1),
    ("Seasonal Differenced: $\\nabla_{12} X_t$", x_diff12),
    ("Moving Average (span=7)", x_ma7),
]):
    freqs_s, Pxx_s = compute_periodogram(series)
    Pxx_smooth = daniell_smooth(Pxx_s, 5)
    ax.plot(freqs_s[1:], Pxx_smooth[1:], linewidth=1.5, color="#2c3e50")
    ax.axvline(1/12, color="red", linestyle="--", alpha=0.5, label="$\\omega=1/12$")
    ax.set_title(f"Spectrum: {label}", fontsize=10)
    ax.set_xlabel("Frequency")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("  Original: low-freq AR peak + seasonal peak at 1/12")
print("  First diff: low-freq power suppressed, seasonal peak remains")
print("  Seasonal diff: seasonal peak removed, low-freq structure remains")
print("  MA(7): high-freq noise suppressed (smoothing effect)")


## **PART 7 — Synthesis (Exam-Ready)**

> **Core points you must be able to state without hesitation:**

---

### **Foundations**

- The spectral density $f(\omega)$ decomposes **variance by frequency**
- The Wiener–Khinchin theorem connects $f(\omega) \leftrightarrow \gamma(h)$ via Fourier transform
- $\int f(\omega)\, d\omega = \gamma(0) = \mathrm{Var}(X_t)$
- A flat spectrum means white noise; peaks indicate periodicities or persistence

### **Estimation**

- The periodogram is the natural estimator of $f(\omega)$
- It is **asymptotically unbiased** but **inconsistent** ($\mathrm{Var}$ does not shrink with $n$)
- Consistent estimation requires **smoothing** (Daniell kernel, Welch, tapering)
- Bandwidth controls the **bias-variance tradeoff**: more smoothing → lower variance but blurred peaks

### **ARMA Spectra**

- ARMA spectral density: $f(\omega) = \sigma^2 |\theta(e^{-2\pi i\omega})|^2 / |\phi(e^{-2\pi i\omega})|^2$
- AR(1) $\phi > 0$: peak at $\omega = 0$ (persistence)
- AR(1) $\phi < 0$: peak at $\omega = 0.5$ (oscillation)
- AR(2) with complex roots: peak at intermediate frequency (pseudo-cycle)
- Seasonal process: discrete peaks at harmonics $\omega = k/s$

### **Filters**

- Linear filters multiply the spectrum by $|A(\omega)|^2$ (power transfer function)
- **First difference** $(1-B)$: high-pass filter, suppresses low frequencies
- **Seasonal difference** $(1-B^s)$: zeros at seasonal harmonics $\omega = k/s$
- **Moving average**: low-pass filter, suppresses high frequencies

### **Connection to Time Domain**

- The ACF and spectral density contain **identical information** (Fourier pair)
- Slow ACF decay ↔ low-frequency spectral peak ↔ persistence
- ACF cutoff at lag $q$ ↔ smooth spectrum (MA structure)
- PACF cutoff at lag $p$ ↔ spectrum determined by AR polynomial


## **PART 8 — Self-Test: Exam Preparation Questions**

> Work through these without looking at notes first.

---

### **Conceptual Questions (Oral Exam Style)**

**Q1.** State the Wiener–Khinchin theorem. What does it tell us about the relationship between the ACF and the spectral density?

**Q2.** Why is the raw periodogram an inconsistent estimator of the spectral density? What does "inconsistent" mean precisely in this context?

**Q3.** What is the bias-variance tradeoff in spectral smoothing? How does the bandwidth parameter control it?

**Q4.** Write the spectral density of AR(1). Sketch its shape for $\phi = 0.9$ and for $\phi = -0.7$. Explain the difference.

**Q5.** Explain how first differencing acts as a filter in the frequency domain. What does $(1-B)$ do to the spectrum?

**Q6.** You observe a spectral peak at $\omega \approx 0.083$ in monthly data. What periodic component does this suggest? What if you also see peaks at $\omega \approx 0.167$ and $\omega \approx 0.25$?

**Q7.** Your colleague claims that because the ACF and spectral density contain the same information, there is no reason to compute the periodogram. Give two situations where the frequency domain view reveals something the ACF does not show clearly.

---

### **Computation Questions**

**Q8.** For AR(1) with $\phi = 0.6$, $\sigma^2 = 1$:
- Compute $f(0)$ and $f(0.5)$
- At what frequency is $f(\omega)$ maximized? What is the maximum value?
- Verify that $\int_{-1/2}^{1/2} f(\omega)\, d\omega = \gamma(0)$ using the known $\gamma(0) = \sigma^2/(1-\phi^2)$

**Q9.** For MA(1) with $\theta = -0.8$, $\sigma^2 = 1$:
- Compute $f(0)$ and $f(0.5)$
- Is the peak at low or high frequency? Why?

**Q10.** The first difference filter has transfer function $A(\omega) = 1 - e^{-2\pi i\omega}$.
- Compute $|A(0)|^2$ and $|A(0.5)|^2$
- Explain why this means differencing removes trend but amplifies noise

---

### **Identification Challenge**

**Q11.** Run the cell below. From the periodograms alone, identify each process.


In [ ]:
# ============================================================
# PART 8: SPECTRAL IDENTIFICATION CHALLENGE
# Four mystery processes. Identify each from the periodogram.
# ============================================================

np.random.seed(777)
n = 512

# Mystery processes
t = np.arange(n)
mysteries = {
    "Mystery A": np.random.normal(0, 1, n),
    "Mystery B": simulate_ar1(0.9, n=n),
    "Mystery C": simulate_ar1(-0.7, n=n),
    "Mystery D": 2.0 * np.sin(2*np.pi*t/20) + 0.5 * np.sin(2*np.pi*t/8) + np.random.normal(0, 0.8, n),
}

fig, axes = plt.subplots(4, 2, figsize=(14, 12))

for row, (label, x) in enumerate(mysteries.items()):
    axes[row, 0].plot(x[:200], linewidth=0.8, color="#2c3e50")
    axes[row, 0].set_title(f"{label} — Time Series", fontsize=11)
    axes[row, 0].grid(True, alpha=0.3)

    freqs_s, Pxx_s = compute_periodogram(x)
    Pxx_sm = daniell_smooth(Pxx_s, 5)
    axes[row, 1].plot(freqs_s[1:], Pxx_sm[1:], linewidth=1.5, color="#2c3e50")
    axes[row, 1].set_title(f"{label} — Smoothed Periodogram", fontsize=11)
    axes[row, 1].set_xlabel("Frequency")
    axes[row, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Answers ---
print("\n" * 2)
print("=" * 60)
print("ANSWERS")
print("=" * 60)
print()
print("Mystery A: White Noise")
print("  -> Flat spectrum, no dominant frequency")
print()
print("Mystery B: AR(1) with phi = 0.9")
print("  -> Strong peak at omega ~ 0 (low-frequency dominance)")
print("  -> Monotonic decrease toward omega = 0.5")
print()
print("Mystery C: AR(1) with phi = -0.7")
print("  -> Peak at omega = 0.5 (high-frequency dominance)")
print("  -> Monotonic increase from omega = 0")
print()
print("Mystery D: Two sinusoids (period=20 and period=8) + noise")
print("  -> Sharp peaks at omega = 1/20 = 0.05 and omega = 1/8 = 0.125")
print("  -> Flat noise floor otherwise")


In [ ]:
# ============================================================
# PART 8: COMPUTATION ANSWERS
# ============================================================

print("=" * 60)
print("Q8: AR(1), phi=0.6, sigma^2=1")
print("=" * 60)
phi, sigma2 = 0.6, 1.0

# f(omega) = sigma^2 / (1 - 2*phi*cos(2*pi*omega) + phi^2)
f_0 = sigma2 / (1 - 2*phi*1 + phi**2)  # cos(0) = 1
f_05 = sigma2 / (1 - 2*phi*(-1) + phi**2)  # cos(pi) = -1
gamma_0 = sigma2 / (1 - phi**2)

print(f"  f(0)   = sigma^2 / (1-phi)^2 = {f_0:.4f}")
print(f"  f(0.5) = sigma^2 / (1+phi)^2 = {f_05:.4f}")
print(f"  Maximum at omega=0 (since phi>0): f(0) = {f_0:.4f}")
print(f"  gamma(0) = sigma^2/(1-phi^2) = {gamma_0:.4f}")
print(f"  (Integral of f(omega) over [-1/2, 1/2] equals gamma(0) by Wiener-Khinchin)")

print(f"\n{'=' * 60}")
print("Q9: MA(1), theta=-0.8, sigma^2=1")
print("=" * 60)
theta, sigma2 = -0.8, 1.0

# f(omega) = sigma^2 * (1 + 2*theta*cos(2*pi*omega) + theta^2)
f_0_ma = sigma2 * (1 + 2*theta + theta**2)  # = (1+theta)^2
f_05_ma = sigma2 * (1 - 2*theta + theta**2)  # = (1-theta)^2

print(f"  f(0)   = sigma^2 * (1+theta)^2 = {f_0_ma:.4f}")
print(f"  f(0.5) = sigma^2 * (1-theta)^2 = {f_05_ma:.4f}")
print(f"  Peak at omega=0.5 because theta<0 (high-frequency amplification)")
print(f"  This matches negative ACF at lag 1: rho(1) = theta/(1+theta^2) = {theta/(1+theta**2):.4f}")

print(f"\n{'=' * 60}")
print("Q10: First difference transfer function")
print("=" * 60)
print(f"  |A(0)|^2   = |1 - e^0|^2 = |1-1|^2 = 0    (completely suppresses DC/trend)")
print(f"  |A(0.5)|^2 = |1 - e^(-i*pi)|^2 = |1-(-1)|^2 = 4  (amplifies Nyquist)")
print(f"  -> Differencing removes trend (zero gain at omega=0)")
print(f"  -> But amplifies high-frequency noise (gain=4 at omega=0.5)")
print(f"  -> This is why overdifferencing is dangerous: it injects noise")
